# 05 - Matched analysis: gestational weight gain and pregnancy outcomes

**Runs in:** Truveta Studio notebook environment only.

**What this notebook does**

This is the analysis notebook. It takes the cohort files produced by notebooks
01-03, defines the exposure groups, and runs the propensity-score-matched
comparisons that the manuscript reports.

**Exposure groups**

| Group | Definition |
|---|---|
| `continued_user` | semaglutide coverage overlapped the pregnancy (any trimester) |
| `former_user` | semaglutide before pregnancy only, no in-pregnancy coverage |
| `non_user` | control cohort (notebook 02) |

People who **initiated** semaglutide during pregnancy are excluded up front
(section 3), because they have no pre-pregnancy treatment period.

**Pairwise comparisons:** `continued vs former`, `continued vs non`, `former vs non`.

**Inputs**

`test_t3.csv`, `control_t1.csv`, `drug_source_label.csv`, `drugexporsure.csv`,
`test_zcodecount.csv`, plus `full_control_df.csv`, `full_med_wo_weight.csv` and
`contrl_zcodecount.csv` for the missing-weight sensitivity analysis.

**Outputs** - everything lands in `revise_results/`:

| File | Contents |
|---|---|
| `<comparison>_matched.csv` | matched analytic dataset for one comparison |
| `<comparison>_balance.csv` | standardised mean differences after matching |
| `<comparison>_logit_results.csv` / `_ols_results.csv` | full model coefficients |
| `<comparison>_*_model_status.csv` | per-outcome fit status (fitted / skipped / error) |
| `pairwise_summary.csv` | sample sizes, caliper and max SMD per comparison |
| `group_effect_only_summary.csv` | the exposure-group effect from every adjusted model |
| `unadjusted_summary_all.csv` | the exposure-group effect from the minimally adjusted models |
| `<comparison>_table1_tableone.csv` | Table 1 for each matched set |
| `csv_outputs/<comparison>_*.csv` | GWG post-hoc chi-square outputs |

**Run order:** 01 -> 02 -> 03 -> 04 -> **05**.

## 1. Setup and configuration

In [ ]:
from truveta.study import Client, OutputMode

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyspark.pandas as ps
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from scipy.special import logit
from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import Table
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

In [ ]:
!pip install tableone
from tableone import TableOne

In [ ]:
# ============================================================================
# CONFIG - edit here, not below
# ============================================================================

POPULATION_TITLE = "Control Group"    # any population works; only the output path is used

# --- input files ----------------------------------------------------------
IN_EXPOSED       = "/test_t3.csv"
IN_CONTROL       = "/control_t1.csv"
IN_SOURCE_LABEL  = "/drug_source_label.csv"
IN_DRUG_EXPOSURE = "/drugexporsure.csv"        # spelling matches notebook 03
IN_DAYSSUPPLY    = "/drug_dayssupply.csv"
IN_ZCODE_EXPOSED = "/test_zcodecount.csv"
IN_ZCODE_CONTROL = "/contrl_zcodecount.csv"
IN_CONTROL_NOWT  = "/full_control_df.csv"
IN_EXPOSED_NOWT  = "/full_med_wo_weight.csv"

RESULTS_SUBDIR = "revise_results"

# --- exposure grouping ----------------------------------------------------
# Trimester boundaries measured from the estimated LMP.
T1_END   = pd.Timedelta(weeks=13, days=6)
T2_START = pd.Timedelta(weeks=14)
T2_END   = pd.Timedelta(weeks=27, days=6)
T3_START = pd.Timedelta(weeks=28)
T3_END   = pd.Timedelta(weeks=40, days=6)

# Map trimester-overlap pattern -> analysis group.
USER_GROUP_MAP = {
    "No_exposure": "former_user",
    "T1_only":     "continued_user",
    "T1_T2":       "continued_user",
    "Throughout":  "continued_user",
}

# --- modelling ------------------------------------------------------------
GROUP_COL = "user_group"

# Covariates entering the propensity score model only.
PS_COVARIATES = [
    "age_at_delivery",
    "race_ethnicity",
    "PrePregnancyBMI",
    "t2d_before_pregnancy",
    "hyper_before_pregnancy",
]

# Extra covariates added to the outcome models on top of the PS covariates.
OUTCOME_EXTRA_COVARIATES = ["prior_Csection", "Prior_Preterm_Birth"]

CONTINUOUS_OUTCOMES = ["gestation_weight"]

BINARY_OUTCOMES = [
    "gwg_excessive_flag",
    "gest_diabetes_no_prior_t2d",
    "preg_related_htn",
    "excessive_fetal_weight",
    "intra_grow_restrict",
    "csection",
    "preterm",
]

PAIRWISE_COMPARISONS = [
    ("continued_user", "former_user", "continued_vs_former"),
    ("continued_user", "non_user",    "continued_vs_non"),
    ("former_user",    "non_user",    "former_vs_non"),
]

CALIPER_MULTIPLIER = 0.2      # caliper = 0.2 x SD of the logit propensity score

In [ ]:
client = Client(output_mode=OutputMode.PandasOnSpark)
study = client.get_study()
population = study.get_population(title=POPULATION_TITLE)
snapshot = population.get_latest_snapshot()

output_path_local = study.get_output_path(fs=True)

output_dir = Path(output_path_local) / RESULTS_SUBDIR
output_dir.mkdir(parents=True, exist_ok=True)
print("results ->", output_dir)

## 2. Load the cohort files

In [ ]:
test = pd.read_csv(output_path_local + IN_EXPOSED)
control = pd.read_csv(output_path_local + IN_CONTROL)
drug_source_label = pd.read_csv(output_path_local + IN_SOURCE_LABEL)
drugexporsure = pd.read_csv(output_path_local + IN_DRUG_EXPOSURE)
test_zcodecount = pd.read_csv(output_path_local + IN_ZCODE_EXPOSED)

test = test.merge(test_zcodecount, on="PersonId")

print("exposed:", test.shape, "| control:", control.shape)
print("source labels:", drug_source_label.shape, "| exposure:", drugexporsure.shape)

In [ ]:
# The control export may carry the legacy column name for prenatal care.
if "obstetric_care" not in control.columns and "Obstetriccare" in control.columns:
    control = control.rename(columns={"Obstetriccare": "obstetric_care"})

## 3. Exclude people who initiated semaglutide during pregnancy

`drug_source_label.csv` only contains people with a pre-pregnancy treatment
episode. Medication users missing from it started treatment during pregnancy and
are dropped: they have no pre-pregnancy exposure window, so they fit neither the
`continued_user` nor the `former_user` definition.

In [ ]:
med_full_df = test[test.source_type == "med"].copy()
surgery_df = test[test.source_type == "surgery"].copy()
print("med:", len(med_full_df), "| surgery:", len(surgery_df))

med_noexppreg = (med_full_df
                 .drop("source_type", axis=1)
                 .merge(drug_source_label, on="PersonId", how="right"))
print(med_noexppreg.source_type.value_counts())

start_drug_preg = med_full_df[~med_full_df.PersonId.isin(med_noexppreg.PersonId)].copy()
print("initiated during pregnancy (dropped):", len(start_drug_preg))

test = test[~test.PersonId.isin(start_drug_preg.PersonId)].copy()
print("exposed cohort after exclusion:", len(test))
print(test.source_type.value_counts())

## 4. Composite hypertensive outcome

`preg_related_htn` = incident gestational hypertension **or** incident
preeclampsia (in both cases only counting people without pre-pregnancy
hypertension).

In [ ]:
for frame in (test, control):
    frame["preg_related_htn"] = (
        (frame["gest_hyper_no_prior_hyper"] == True) |
        (frame["preeclampsia_no_prior_hyper"] == True)
    )

print(test["preg_related_htn"].value_counts())
print(control["preg_related_htn"].value_counts())

In [ ]:
# Sanity check: the two cohorts must not share people.
overlap = set(test["PersonId"]).intersection(set(control["PersonId"]))
print("overlapping PersonIds:", len(overlap))

## 5. Gestational weight gain category (IOM / NASEM 2009)

Observed `gestation_weight` is compared to the recommended range for the
person's pre-pregnancy BMI category at their gestational age at delivery:

* first-trimester allowance of 0.5-2.0 kg, plus
* a BMI-specific weekly rate applied from week 13 onwards.

In [ ]:
def bmi_category(bmi):
    """WHO pre-pregnancy BMI category."""
    if pd.isna(bmi):
        return np.nan
    if bmi < 18.5:
        return "Underweight"
    if bmi < 25.0:
        return "Normal weight"
    if bmi < 30.0:
        return "Overweight"
    return "Obese"


def gwg_bounds(bmi, weeks):
    """Recommended (low, high) total gestational weight gain in kg."""
    if pd.isna(bmi) or pd.isna(weeks):
        return (np.nan, np.nan)

    delta_weeks = max(weeks - 13, 0)   # no weekly accrual before week 13

    if bmi < 18.5:                     # Underweight
        return (0.5 + 0.44 * delta_weeks, 2.0 + 0.58 * delta_weeks)
    if bmi < 25.0:                     # Normal weight
        return (0.5 + 0.35 * delta_weeks, 2.0 + 0.50 * delta_weeks)
    if bmi < 30.0:                     # Overweight
        return (0.5 + 0.23 * delta_weeks, 2.0 + 0.33 * delta_weeks)
    return (0.5 + 0.17 * delta_weeks, 2.0 + 0.27 * delta_weeks)   # Obese


def categorize_gwg(gwg, low, high):
    """Classify observed gestational weight gain against the recommended band."""
    if pd.isna(gwg) or pd.isna(low) or pd.isna(high):
        return np.nan
    if gwg < low:
        return "Inadequate"
    if gwg <= high:
        return "Adequate"
    return "Excessive"


def add_gwg_category(df, bmi_col="PrePregnancyBMI",
                     weeks_col="gestational_week", gwg_col="gestation_weight"):
    """Add `bmi_category`, the recommended bounds, and `gwg_category`."""
    df = df.copy()
    df["bmi_category"] = df[bmi_col].apply(bmi_category)

    bounds = df.apply(lambda row: gwg_bounds(row[bmi_col], row[weeks_col]), axis=1)
    df[["gwg_low_bound", "gwg_high_bound"]] = pd.DataFrame(bounds.tolist(), index=df.index)

    df["gwg_category"] = df.apply(
        lambda row: categorize_gwg(row[gwg_col], row["gwg_low_bound"], row["gwg_high_bound"]),
        axis=1)
    df["gwg_category"] = pd.Categorical(
        df["gwg_category"], categories=["Inadequate", "Adequate", "Excessive"], ordered=True)
    return df


test = add_gwg_category(test)
control = add_gwg_category(control)

print(test.bmi_category.value_counts())
print(test.gwg_category.value_counts())
print(control.gwg_category.value_counts())

In [ ]:
# Optional: write the categories back into the cohort files so other notebooks
# see them too. Safe to skip - this notebook recomputes them on every run.
#
# test.to_csv(output_path_local + IN_EXPOSED, index=False)
# control.to_csv(output_path_local + IN_CONTROL, index=False)

## 6. Exposure timing during pregnancy

A person is "exposed" in a trimester if their treatment coverage window
(`DrugStart` -> `DrugEnd` from notebook 03) overlaps that trimester at all.
The overlap pattern is collapsed into `pregnancy_exposure_group`, and then into
the two-level `user_group` used for matching.

In [ ]:
test_df = test[test.source_type == "med"].copy()
test_df = test_df.merge(drugexporsure, on="PersonId", how="left")
print(test_df.shape)
print(test_df.ExposedInPregnancy.value_counts())

In [ ]:
df = test_df.copy()

for c in ["estimated_LMP", "DrugStart", "DrugEnd", "delivery_date"]:
    df[c] = pd.to_datetime(df[c])

# Trimester boundaries for each pregnancy.
df["T1_end"]   = df["estimated_LMP"] + T1_END
df["T2_start"] = df["estimated_LMP"] + T2_START
df["T2_end"]   = df["estimated_LMP"] + T2_END
df["T3_start"] = df["estimated_LMP"] + T3_START
df["T3_end"]   = df["estimated_LMP"] + T3_END

# Overlap between the treatment window and each trimester.
df["exposed_T1"] = (df["DrugStart"] <= df["T1_end"])   & (df["DrugEnd"] >= df["estimated_LMP"])
df["exposed_T2"] = (df["DrugStart"] <= df["T2_end"])   & (df["DrugEnd"] >= df["T2_start"])
df["exposed_T3"] = (df["DrugStart"] <= df["T3_end"])   & (df["DrugEnd"] >= df["T3_start"])

In [ ]:
def classify_exposure(row):
    """Collapse the three trimester flags into a single exposure pattern."""
    t1, t2, t3 = row["exposed_T1"], row["exposed_T2"], row["exposed_T3"]

    if t1 and not t2 and not t3:
        return "T1_only"
    if t1 and t2 and not t3:
        return "T1_T2"
    if t1 and t2 and t3:
        return "Throughout"
    if not t1 and t2 and not t3:
        return "T2_only"
    if not t1 and not t2 and t3:
        return "T3_only"
    if not t1 and t2 and t3:
        return "T2_T3"
    return "No_exposure"


df["pregnancy_exposure_group"] = df.apply(classify_exposure, axis=1)
df["user_group"] = df["pregnancy_exposure_group"].replace(USER_GROUP_MAP)

print(df.pregnancy_exposure_group.value_counts())
print(df.user_group.value_counts())

In [ ]:
# Continued users: did they refill after conception, or just carry over supply?
df_preg = df[df["user_group"] == "continued_user"].copy()
df_preg["new_refill"] = df_preg["NumDrugEpisodes"] > 1
print(df_preg["new_refill"].value_counts())

# Former users: how long between the end of supply and conception?
df_former = df[df["user_group"] == "former_user"].copy()
df_former["days_between_stop_and_preg"] = (
    df_former["estimated_LMP"] - df_former["DrugEnd"]).dt.days
df_former["days_between_stop_and_preg"].describe()

In [ ]:
# Days of supply per fill. `supply_days` is the treatment-window length implied
# by DrugStart/DrugEnd; `Supply` is the median dispensed days supply.
df["supply_days"] = (pd.to_datetime(df["DrugEnd"]) - pd.to_datetime(df["DrugStart"])).dt.days

print(df.groupby("pregnancy_exposure_group").agg(
    mean_supply=("supply_days", "mean"),
    std_supply=("supply_days", "std"),
    median_supply=("supply_days", "median")))

daysupp = pd.read_csv(output_path_local + IN_DAYSSUPPLY)
df["PersonId"] = df["PersonId"].astype(str)
daysupp["PersonId"] = daysupp["PersonId"].astype(str)
df = df.merge(daysupp[["PersonId", "Supply"]], on="PersonId", how="left")
df.Supply.describe()

In [ ]:
TableOne(df, columns=["Supply"], categorical=[],
         groupby="pregnancy_exposure_group", nonnormal=["Supply"], pval=True)

In [ ]:
mode_week = test_df["gestational_week"].round().mode()[0]
median_week = test_df["gestational_week"].round().median()
plt.figure()
test_df["gestational_week"].hist(bins=40)
plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")
plt.axvline(median_week, linestyle=":", label=f"Median week: {median_week}")
plt.xlabel("Gestational age (weeks)")
plt.ylabel("Count")
plt.title("Medication group: gestational age at delivery")
plt.legend()
plt.show()

## 7. Control group

Controls need a pre-pregnancy BMI (it is a matching covariate), so those without
one are dropped.

In [ ]:
control_df = control.copy()
print("control:", control_df.shape)
control_df = control_df.dropna(subset=["PrePregnancyBMI"]).copy()
print("with PrePregnancyBMI:", control_df.shape)

control_df["user_group"] = "non_user"

In [ ]:
mode_week = control_df["gestational_week"].round().mode()[0]
median_week = control_df["gestational_week"].round().median()
plt.figure()
control_df["gestational_week"].hist(bins=40)
plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")
plt.axvline(median_week, linestyle=":", label=f"Median week: {median_week}")
plt.xlabel("Gestational age (weeks)")
plt.ylabel("Count")
plt.title("Control group: gestational age at delivery")
plt.legend()
plt.show()

## 8. Combined analytic dataset

Treatment-specific columns (drug persistence, weight loss, etc.) do not exist
for controls; they are created as all-`NaN` so the two frames stack cleanly.

In [ ]:
ANALYSIS_COLS = [
    # exposure
    "pregnancy_exposure_group", "user_group",
    # demographics
    "age_at_delivery", "race_ethnicity", "Income", "parity",
    # BMI / weight
    "preTreatmentBMI", "PrePregnancyBMI", "weight_loss",
    # treatment
    "med_indict", "AccumulatedPersistenceBeforePregnancy",
    "TotalExposureDaysInPregnancy", "supply_days",
    # comorbidity / history
    "t2d_before_pregnancy", "hyper_before_pregnancy", "depression",
    "prior_Csection", "Prior_Preterm_Birth", "obstetric_care", "zcode_count",
    # outcomes
    "gestational_week", "gestation_weight", "gwg_category",
    "gest_diabetes_no_prior_t2d", "preg_related_htn", "excessive_fetal_weight",
    "intra_grow_restrict", "csection", "preterm",
]


def ensure_columns(df, columns):
    """Add any missing columns as all-NaN so frames can be concatenated."""
    df = df.copy()
    for c in columns:
        if c not in df.columns:
            df[c] = np.nan
    return df


control_df = ensure_columns(control_df, ANALYSIS_COLS)

df_treated_sub = df[ANALYSIS_COLS].copy()
df_control_sub = control_df[ANALYSIS_COLS].copy()
df_all = pd.concat([df_treated_sub, df_control_sub], ignore_index=True)

print(df_all.user_group.value_counts())

In [ ]:
# Binary version of the primary GWG outcome.
df_all["gwg_excessive_flag"] = (df_all["gwg_category"] == "Excessive").map(
    {True: "yes", False: "no"})
df_all["gwg_excessive_flag"].value_counts(dropna=False)

## 9. Propensity score matching

**Design.** For each pairwise comparison, fit a logistic propensity score on
`PS_COVARIATES`, then perform 1:1 greedy nearest-neighbour matching **without
replacement** on the logit of the propensity score, with a caliper of
`CALIPER_MULTIPLIER` x SD(logit PS).

Balance is reported as absolute standardised mean differences (SMD); < 0.1 is
conventionally taken as adequate.

**Outcome models.** On the matched set, logistic regression for binary outcomes
and OLS for continuous ones, adjusted for the PS covariates plus income, parity,
depression, prior C-section and prior preterm birth. `t2d_before_pregnancy` is
dropped from the gestational diabetes model and `hyper_before_pregnancy` from
the hypertensive models, since those outcomes are defined as "incident, no prior
diagnosis" and the covariate would be constant.

In [ ]:
# --- small utilities ------------------------------------------------------

def clean_binary_series(s):
    """Coerce a yes/no/true/false/1/0 column to numeric 0/1."""
    if s.dtype == bool:
        return s.astype(int)

    mapping = {"yes": 1, "no": 0, "y": 1, "n": 0, "true": 1, "false": 0,
               "t": 1, "f": 0, "case": 1, "control": 0,
               "positive": 1, "negative": 0}

    if s.dtype == object or str(s.dtype).startswith("string"):
        s2 = s.astype(str).str.strip().str.lower().replace(mapping)
        return pd.to_numeric(s2, errors="coerce")

    return pd.to_numeric(s, errors="coerce")


def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")

In [ ]:
# --- build the pairwise dataset ------------------------------------------

# Columns carried through matching purely so Table 1 can report them.
EXTRA_KEEP_COLS = [
    "pregnancy_exposure_group", "preTreatmentBMI", "weight_loss", "med_indict",
    "AccumulatedPersistenceBeforePregnancy", "TotalExposureDaysInPregnancy",
    "supply_days", "obstetric_care", "zcode_count", "gestational_week", "gwg_category",
]

ALL_OUTCOMES = CONTINUOUS_OUTCOMES + BINARY_OUTCOMES


def prepare_pairwise_data(df_all, group_col, group1, group0, covariates,
                          outcome_extra_covariates, outcomes, extra_keep_cols=None):
    """Subset to two groups and keep only the columns the pipeline needs.

    `treat` = 1 for `group1`, 0 for `group0`.
    """
    df_sub = df_all[df_all[group_col].isin([group1, group0])].copy()
    if df_sub.empty:
        return pd.DataFrame()

    df_sub["treat"] = (df_sub[group_col] == group1).astype(int)

    id_fields = ["PersonId"] if "PersonId" in df_sub.columns else []

    needed = list(dict.fromkeys(
        id_fields + [group_col, "treat"] + covariates + outcome_extra_covariates +
        outcomes + ["Income", "parity", "depression"] +
        (extra_keep_cols if extra_keep_cols is not None else [])))

    df_sub = ensure_columns(df_sub, needed)[needed].copy()
    return df_sub.reset_index(drop=False).rename(columns={"index": "orig_index"})

In [ ]:
def build_ps_design_matrix(df_sub, covariates):
    """Numeric design matrix for the PS model.

    Race/ethnicity is one-hot encoded; every other covariate is coerced to
    numeric with median imputation (all-missing columns become 0).
    """
    X = df_sub[covariates].copy()

    if "race_ethnicity" in X.columns:
        X["race_ethnicity"] = X["race_ethnicity"].fillna("Unknown").astype(str)

    for c in X.columns:
        if c == "race_ethnicity":
            continue
        X[c] = X[c].astype(int) if X[c].dtype == bool else pd.to_numeric(X[c], errors="coerce")

    for c in [c for c in X.columns if c != "race_ethnicity"]:
        X[c] = 0 if X[c].isna().all() else X[c].fillna(X[c].median())

    if "race_ethnicity" in X.columns:
        X = pd.get_dummies(X, columns=["race_ethnicity"], drop_first=True)

    for c in X.columns:
        if X[c].dtype == bool:
            X[c] = X[c].astype(int)
        X[c] = pd.to_numeric(X[c], errors="coerce")
        X[c] = 0 if X[c].isna().all() else X[c].fillna(X[c].median())

    return X


def run_ps(df_sub, covariates):
    """Fit the propensity score model and attach `ps` to the frame."""
    if df_sub.empty:
        raise ValueError("run_ps received an empty dataframe.")

    X = build_ps_design_matrix(df_sub, covariates)
    y = df_sub["treat"].copy()

    valid_mask = np.isfinite(X).all(axis=1)
    X, y, df_sub = X.loc[valid_mask].copy(), y.loc[valid_mask].copy(), df_sub.loc[valid_mask].copy()

    if X.empty:
        raise ValueError("No rows remain after building the PS design matrix.")
    if y.nunique() < 2:
        raise ValueError("Treatment indicator has fewer than 2 classes after filtering.")

    model = LogisticRegression(max_iter=2000).fit(X, y)

    df_sub = df_sub.copy()
    df_sub["ps"] = model.predict_proba(X)[:, 1]
    return df_sub, X.columns.tolist(), model

In [ ]:
def compute_caliper_from_logit_ps(df_sub, multiplier=CALIPER_MULTIPLIER):
    """Caliper = multiplier x SD of the logit propensity score."""
    ps_clipped = df_sub["ps"].clip(1e-6, 1 - 1e-6)
    return multiplier * np.std(logit(ps_clipped))


def match_ps_without_replacement(df_sub, caliper=None, caliper_type="logit_ps"):
    """Greedy 1:1 nearest-neighbour matching without replacement.

    Treated units are processed in ascending propensity-score order; each takes
    the closest remaining control, provided the distance is within `caliper`.
    """
    if df_sub.empty:
        return pd.DataFrame()

    treated = df_sub[df_sub["treat"] == 1].copy().sort_values("ps")
    control = df_sub[df_sub["treat"] == 0].copy().sort_values("ps")

    if treated.empty or control.empty:
        return pd.DataFrame()

    if caliper_type == "logit_ps":
        treated["match_score"] = logit(treated["ps"].clip(1e-6, 1 - 1e-6))
        control["match_score"] = logit(control["ps"].clip(1e-6, 1 - 1e-6))
    else:
        treated["match_score"] = treated["ps"]
        control["match_score"] = control["ps"]

    control_available = control.copy()
    matched_rows, pair_id = [], 0

    for _, t_row in treated.iterrows():
        if control_available.empty:
            break

        control_available = control_available.copy()
        control_available["score_diff"] = (
            control_available["match_score"] - t_row["match_score"]).abs()

        best_idx = control_available["score_diff"].idxmin()
        best_diff = control_available.loc[best_idx, "score_diff"]

        if (caliper is None) or (best_diff <= caliper):
            t_out, c_out = t_row.copy(), control_available.loc[best_idx].copy()
            t_out["pair_id"] = c_out["pair_id"] = pair_id
            matched_rows.extend([t_out, c_out])

            control_available = control_available.drop(index=best_idx)
            pair_id += 1

    matched_df = pd.DataFrame(matched_rows).reset_index(drop=True)
    return matched_df.drop(columns=["score_diff"], errors="ignore")

In [ ]:
# --- balance diagnostics --------------------------------------------------

def smd_continuous(x_t, x_c):
    """Absolute standardised mean difference for a continuous variable."""
    x_t = pd.to_numeric(pd.Series(x_t), errors="coerce").dropna()
    x_c = pd.to_numeric(pd.Series(x_c), errors="coerce").dropna()
    if len(x_t) == 0 or len(x_c) == 0:
        return np.nan

    var_t = np.var(x_t, ddof=1) if len(x_t) > 1 else 0
    var_c = np.var(x_c, ddof=1) if len(x_c) > 1 else 0
    pooled_sd = np.sqrt((var_t + var_c) / 2)

    if pooled_sd == 0 or np.isnan(pooled_sd):
        return 0.0
    return abs(np.mean(x_t) - np.mean(x_c)) / pooled_sd


def smd_binary(x_t, x_c):
    """Absolute standardised mean difference for a 0/1 variable."""
    x_t = pd.to_numeric(pd.Series(x_t), errors="coerce").dropna()
    x_c = pd.to_numeric(pd.Series(x_c), errors="coerce").dropna()
    if len(x_t) == 0 or len(x_c) == 0:
        return np.nan

    p1, p0 = np.mean(x_t), np.mean(x_c)
    denom = np.sqrt((p1 * (1 - p1) + p0 * (1 - p0)) / 2)

    if denom == 0 or np.isnan(denom):
        return 0.0
    return abs(p1 - p0) / denom


def get_balance_table(matched_df, covariates):
    """SMD for every column of the PS design matrix, worst first."""
    if matched_df.empty:
        return pd.DataFrame(columns=["variable", "SMD"])

    X = build_ps_design_matrix(matched_df, covariates)
    balance_df = pd.concat(
        [matched_df[["treat"]].reset_index(drop=True), X.reset_index(drop=True)], axis=1)

    treated = balance_df[balance_df["treat"] == 1]
    control = balance_df[balance_df["treat"] == 0]

    rows = []
    for col in X.columns:
        vals = balance_df[col].dropna().unique()
        if len(vals) == 0:
            smd_val = np.nan
        elif set(vals).issubset({0, 1}):
            smd_val = smd_binary(treated[col], control[col])
        else:
            smd_val = smd_continuous(treated[col], control[col])
        rows.append({"variable": col, "SMD": smd_val})

    return pd.DataFrame(rows).sort_values("SMD", ascending=False).reset_index(drop=True)

In [ ]:
# --- prepare the matched set for modelling -------------------------------

RACE_LEVELS   = ["Non-Hispanic White", "Non-Hispanic Black", "Hispanic", "Other", "Unknown"]
INCOME_LEVELS = ["≤50000", "50001-80000", ">80000", "Unknown"]
PARITY_LEVELS = ["Primiparous", "Multiparous", "Unknown"]


def prepare_matched_for_models(matched_df, treat_label, ref_label):
    """Recode the matched frame: readable group labels, fixed category orders,
    numeric outcomes and covariates.
    """
    if matched_df.empty:
        return matched_df.copy()

    df = matched_df.copy()
    df["group"] = np.where(df["treat"] == 1, treat_label, ref_label)

    for col, levels in [("race_ethnicity", RACE_LEVELS),
                        ("Income", INCOME_LEVELS),
                        ("parity", PARITY_LEVELS)]:
        if col in df.columns:
            df[col] = pd.Categorical(df[col].fillna("Unknown"),
                                     categories=levels, ordered=True)

    # Reference level first, so model coefficients read as "treat vs ref".
    df["group"] = pd.Categorical(df["group"], categories=[ref_label, treat_label], ordered=True)

    for col in BINARY_OUTCOMES:
        if col in df.columns:
            df[col] = clean_binary_series(df[col])

    for col in ["age_at_delivery", "PrePregnancyBMI", "t2d_before_pregnancy",
                "hyper_before_pregnancy", "depression",
                "prior_Csection", "Prior_Preterm_Birth"]:
        if col in df.columns:
            df[col] = safe_numeric(df[col])

    return df

In [ ]:
# --- outcome models -------------------------------------------------------

def get_covariates_for_outcome(outcome, df_subset):
    """Adjustment set for one outcome, dropping conditioned-out and constant terms."""
    covs = ["C(group)", "age_at_delivery", "C(race_ethnicity)", "C(Income)",
            "PrePregnancyBMI", "C(parity)", "t2d_before_pregnancy",
            "hyper_before_pregnancy", "depression",
            "prior_Csection", "Prior_Preterm_Birth"]

    # These outcomes are defined as "no prior diagnosis", so the corresponding
    # history covariate carries no information.
    if outcome == "gest_diabetes_no_prior_t2d":
        covs = [c for c in covs if "t2d_before_pregnancy" not in c]
    if outcome in ("gest_hyper_no_prior_hyper", "preg_related_htn"):
        covs = [c for c in covs if "hyper_before_pregnancy" not in c]

    valid = []
    for cov in covs:
        colname = cov.split("(")[-1].split(")")[0] if "C(" in cov else cov
        if colname in df_subset.columns and df_subset[colname].nunique(dropna=True) > 1:
            valid.append(cov)
    return valid


def extract_model_results(model, outcome_name, comparison_name,
                          model_type="logit", fit_engine=None):
    """Tidy a fitted model into one row per coefficient, with a formatted string."""
    conf_int = model.conf_int()
    summary_df = pd.DataFrame({
        "variable": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "z_or_t": model.tvalues if hasattr(model, "tvalues") else model.params / model.bse,
        "p_value": model.pvalues.values,
        "ci_lower": conf_int[0].values,
        "ci_upper": conf_int[1].values,
    })

    if model_type == "logit":
        summary_df["odds_ratio"] = np.exp(summary_df["coef"])
        summary_df["or_ci_lower"] = np.exp(summary_df["ci_lower"])
        summary_df["or_ci_upper"] = np.exp(summary_df["ci_upper"])
        summary_df["formatted"] = summary_df.apply(
            lambda r: f"OR={r['odds_ratio']:.2f}, ({r['or_ci_lower']:.2f}, {r['or_ci_upper']:.2f}), p={r['p_value']:.3f}",
            axis=1)
    else:
        summary_df["formatted"] = summary_df.apply(
            lambda r: f"β={r['coef']:.2f}, ({r['ci_lower']:.2f}, {r['ci_upper']:.2f}), p={r['p_value']:.3f}",
            axis=1)

    summary_df["outcome"] = outcome_name
    summary_df["comparison"] = comparison_name
    summary_df["model_type"] = model_type
    summary_df["fit_engine"] = fit_engine
    return summary_df

In [ ]:
def run_models(df, outcomes, comparison_name, model_type="logit",
               output_csv=None, status_csv=None):
    """Fit one model per outcome and record why any were skipped.

    Returns `(results_df, status_df)`. Logit falls back to a binomial GLM if
    `smf.logit` fails to converge.
    """
    all_results, model_status_rows = [], []

    if df.empty:
        status_df = pd.DataFrame([{
            "comparison": comparison_name, "outcome": None, "model_type": model_type,
            "status": "skipped_empty_input_df", "n": 0, "n_event": np.nan,
            "message": "Input dataframe is empty"}])
        if status_csv is not None:
            status_df.to_csv(status_csv, index=False)
        final_df = pd.DataFrame()
        if output_csv is not None:
            final_df.to_csv(output_csv, index=False)
        return final_df, status_df

    for outcome in outcomes:
        if outcome not in df.columns:
            model_status_rows.append({
                "comparison": comparison_name, "outcome": outcome, "model_type": model_type,
                "status": "skipped_missing_column", "n": np.nan, "n_event": np.nan,
                "message": "Outcome column not found"})
            continue

        df_subset = df.copy()
        covariates = get_covariates_for_outcome(outcome, df_subset)

        cols_needed = [outcome] + [
            c.split("(")[-1].split(")")[0] if "C(" in c else c for c in covariates]
        cols_needed = [c for c in cols_needed if c in df_subset.columns]
        df_subset = df_subset.dropna(subset=cols_needed).copy()

        if outcome in BINARY_OUTCOMES:
            df_subset[outcome] = clean_binary_series(df_subset[outcome])

        if df_subset.empty:
            model_status_rows.append({
                "comparison": comparison_name, "outcome": outcome, "model_type": model_type,
                "status": "skipped_empty_after_dropna", "n": 0, "n_event": np.nan,
                "message": "No rows left after dropna"})
            continue

        n_total = len(df_subset)
        n_event = df_subset[outcome].sum() if outcome in BINARY_OUTCOMES else np.nan

        if df_subset[outcome].nunique(dropna=True) < 2:
            model_status_rows.append({
                "comparison": comparison_name, "outcome": outcome, "model_type": model_type,
                "status": "skipped_no_variation", "n": n_total, "n_event": n_event,
                "message": "Outcome has <2 unique values"})
            continue

        formula = f"{outcome} ~ " + (" + ".join(covariates) if covariates else "1")

        try:
            if model_type == "logit":
                try:
                    model = smf.logit(formula=formula, data=df_subset).fit(disp=False)
                    fit_engine = "logit"
                except Exception:
                    model = smf.glm(formula=formula, data=df_subset,
                                    family=sm.families.Binomial()).fit()
                    fit_engine = "glm_binomial"
            elif model_type == "ols":
                model = smf.ols(formula=formula, data=df_subset).fit()
                fit_engine = "ols"
            else:
                raise ValueError("Unsupported model_type")

            all_results.append(extract_model_results(
                model, outcome, comparison_name, model_type, fit_engine))

            model_status_rows.append({
                "comparison": comparison_name, "outcome": outcome, "model_type": model_type,
                "status": "fitted", "n": n_total, "n_event": n_event, "message": fit_engine})

        except Exception as e:
            model_status_rows.append({
                "comparison": comparison_name, "outcome": outcome, "model_type": model_type,
                "status": "error", "n": n_total, "n_event": n_event, "message": str(e)})

    final_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
    status_df = pd.DataFrame(model_status_rows)

    if output_csv is not None:
        final_df.to_csv(output_csv, index=False)
    if status_csv is not None:
        status_df.to_csv(status_csv, index=False)

    return final_df, status_df

In [ ]:
# --- run one comparison end to end ---------------------------------------

def empty_result_package(label, group1, group0, message):
    """Placeholder result so a failed comparison does not break the loop."""
    return {
        "df_ps": pd.DataFrame(), "matched_df": pd.DataFrame(),
        "matched_model_df": pd.DataFrame(), "balance_table": pd.DataFrame(),
        "logit_results": pd.DataFrame(), "ols_results": pd.DataFrame(),
        "logit_status": pd.DataFrame(), "ols_status": pd.DataFrame(),
        "summary": {"comparison": label, "group1": group1, "group0": group0,
                    "n_before_treated": 0, "n_before_control": 0,
                    "n_after_treated": 0, "n_after_control": 0, "n_pairs": 0,
                    "caliper": np.nan, "max_smd": np.nan, "message": message},
    }


def run_pair_analysis(df_all, group1, group0, label):
    """Subset -> propensity score -> match -> balance -> outcome models."""
    df_sub = prepare_pairwise_data(
        df_all, GROUP_COL, group1, group0, PS_COVARIATES,
        OUTCOME_EXTRA_COVARIATES, ALL_OUTCOMES, extra_keep_cols=EXTRA_KEEP_COLS)

    if df_sub.empty:
        return empty_result_package(label, group1, group0, "No rows after pairwise subsetting")

    try:
        df_ps, ps_features, ps_model = run_ps(df_sub, PS_COVARIATES)
    except Exception as e:
        return empty_result_package(label, group1, group0, f"PS model failed: {e}")

    caliper = compute_caliper_from_logit_ps(df_ps, multiplier=CALIPER_MULTIPLIER)
    matched_df = match_ps_without_replacement(df_ps, caliper=caliper, caliper_type="logit_ps")

    if matched_df.empty:
        return empty_result_package(label, group1, group0, "No matched pairs found")

    balance_tbl = get_balance_table(matched_df, PS_COVARIATES)
    matched_model_df = prepare_matched_for_models(matched_df, treat_label=group1, ref_label=group0)

    logit_results, logit_status = run_models(
        matched_model_df, BINARY_OUTCOMES, comparison_name=label, model_type="logit",
        output_csv=output_dir / f"{label}_logit_results.csv",
        status_csv=output_dir / f"{label}_logit_model_status.csv")

    ols_results, ols_status = run_models(
        matched_model_df, CONTINUOUS_OUTCOMES, comparison_name=label, model_type="ols",
        output_csv=output_dir / f"{label}_ols_results.csv",
        status_csv=output_dir / f"{label}_ols_model_status.csv")

    balance_tbl.to_csv(output_dir / f"{label}_balance.csv", index=False)
    matched_model_df.to_csv(output_dir / f"{label}_matched.csv", index=False)

    summary = {
        "comparison": label, "group1": group1, "group0": group0,
        "n_before_treated": int((df_ps["treat"] == 1).sum()),
        "n_before_control": int((df_ps["treat"] == 0).sum()),
        "n_after_treated": int((matched_df["treat"] == 1).sum()),
        "n_after_control": int((matched_df["treat"] == 0).sum()),
        "n_pairs": int(matched_df["pair_id"].nunique()) if "pair_id" in matched_df.columns else 0,
        "caliper": caliper,
        "max_smd": balance_tbl["SMD"].max() if len(balance_tbl) > 0 else np.nan,
    }

    return {"df_ps": df_ps, "matched_df": matched_df, "matched_model_df": matched_model_df,
            "balance_table": balance_tbl, "logit_results": logit_results,
            "ols_results": ols_results, "logit_status": logit_status,
            "ols_status": ols_status, "summary": summary}

In [ ]:
missing_from_df_all = [c for c in EXTRA_KEEP_COLS if c not in df_all.columns]
if missing_from_df_all:
    print("These columns are missing from df_all and will be all-NaN in the matched output:")
    print(missing_from_df_all)
else:
    print("All extra_keep_cols are present in df_all.")

analysis_results = {}
for group1, group0, label in PAIRWISE_COMPARISONS:
    print(f"\nRunning: {label}")
    analysis_results[label] = run_pair_analysis(df_all, group1, group0, label)

summary_df = pd.DataFrame([v["summary"] for v in analysis_results.values()])
summary_df.to_csv(output_dir / "pairwise_summary.csv", index=False)

print("\n=== Pairwise summary ===")
print(summary_df)

In [ ]:
def extract_group_effect_only(results_df):
    """Keep just the exposure-group coefficient rows."""
    if results_df.empty:
        return results_df

    out = results_df[results_df["variable"].str.contains(r"C\(group\)", regex=True, na=False)].copy()
    keep_cols = [c for c in ["comparison", "outcome", "variable", "formatted",
                             "model_type", "fit_engine", "p_value"] if c in out.columns]
    return out[keep_cols]


group_effect_tables = []
for label, res in analysis_results.items():
    for key in ("logit_results", "ols_results"):
        if not res[key].empty:
            group_effect_tables.append(extract_group_effect_only(res[key]))

group_effect_table = (pd.concat(group_effect_tables, ignore_index=True)
                      if group_effect_tables else pd.DataFrame())
group_effect_table.to_csv(output_dir / "group_effect_only_summary.csv", index=False)
group_effect_table

In [ ]:
# Combined fit-status log across every comparison, so skipped outcomes are visible.
all_status = []
for label, res in analysis_results.items():
    for key in ("logit_status", "ols_status"):
        if key in res and not res[key].empty:
            all_status.append(res[key])

model_status_df = pd.concat(all_status, ignore_index=True) if all_status else pd.DataFrame()
model_status_df.to_csv(output_dir / "all_model_status_summary.csv", index=False)

print("Results saved to:", output_dir)
model_status_df

## 10. Minimally adjusted models

Same matched sets, but the only covariates besides the exposure group are prior
C-section and prior preterm birth. The reference group is set explicitly per
comparison.

> **Naming note.** In the original run this table was written to *both*
> `adjusted_summary_all.csv` and `unadjusted_summary_all.csv`. Both files hold
> the same minimally adjusted estimates. The fully adjusted estimates are the
> ones in `group_effect_only_summary.csv` from section 9.

In [ ]:
MATCHED_FILES = {
    "continued_vs_former": "continued_vs_former_matched.csv",
    "continued_vs_non":    "continued_vs_non_matched.csv",
    "former_vs_non":       "former_vs_non_matched.csv",
}

REFERENCE_MAP = {
    "continued_vs_former": "former_user",
    "continued_vs_non":    "non_user",
    "former_vs_non":       "non_user",
}

TREAT_MAP = {
    "continued_vs_former": "continued_user",
    "continued_vs_non":    "continued_user",
    "former_vs_non":       "former_user",
}

MINIMAL_COVARIATES = ["prior_Csection", "Prior_Preterm_Birth"]

In [ ]:
def extract_adjusted_results(model, outcome_name, comparison_name, model_type):
    """Tidy a minimally adjusted model into one row per coefficient."""
    result_df = pd.DataFrame({
        "variable": model.params.index,
        "coef": model.params.values,
        "p_value": model.pvalues.values,
        "ci_lower": model.conf_int()[0].values,
        "ci_upper": model.conf_int()[1].values,
    })

    if model_type == "logit":
        result_df["odds_ratio"] = np.exp(result_df["coef"])
        result_df["or_ci_lower"] = np.exp(result_df["ci_lower"])
        result_df["or_ci_upper"] = np.exp(result_df["ci_upper"])
        result_df["formatted"] = result_df.apply(
            lambda r: f"OR={r['odds_ratio']:.2f}, ({r['or_ci_lower']:.2f}, {r['or_ci_upper']:.2f}), p={r['p_value']:.3f}",
            axis=1)
    else:
        result_df["formatted"] = result_df.apply(
            lambda r: f"β={r['coef']:.2f}, ({r['ci_lower']:.2f}, {r['ci_upper']:.2f}), p={r['p_value']:.3f}",
            axis=1)

    result_df["outcome"] = outcome_name
    result_df["comparison"] = comparison_name
    result_df["model_type"] = f"{model_type}_adjusted"
    return result_df


def minimal_covariates_for(df):
    """Keep only the minimal covariates that actually vary in this frame."""
    return [c for c in MINIMAL_COVARIATES
            if c in df.columns and df[c].nunique(dropna=True) > 1]

In [ ]:
def run_minimal_models(df, label, ref_group, treat_group=None):
    """Fit group + prior-history models for every outcome in one matched set."""
    if "group" not in df.columns:
        raise ValueError(f"[{label}] missing 'group' column")

    df = ensure_columns(df.copy(), MINIMAL_COVARIATES)
    df["group"] = df["group"].astype(str).str.strip()

    for c in MINIMAL_COVARIATES:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if treat_group is not None:
        df["group"] = pd.Categorical(df["group"],
                                     categories=[ref_group, treat_group], ordered=True)
    else:
        df["group"] = df["group"].astype("category")

    group_term = f"C(group, Treatment(reference='{ref_group}'))"
    all_results = []

    for outcome, model_type in ([(o, "logit") for o in BINARY_OUTCOMES] +
                                [(o, "ols") for o in CONTINUOUS_OUTCOMES]):
        if outcome not in df.columns:
            print(f"[{label}] Skipped {outcome} (not found)")
            continue

        covariates = minimal_covariates_for(df)
        df_sub = df[[outcome, "group"] + covariates].copy()
        df_sub[outcome] = pd.to_numeric(df_sub[outcome], errors="coerce")
        df_sub = df_sub.dropna()

        if df_sub.empty or df_sub[outcome].nunique() < 2:
            print(f"[{label}] Skipped {outcome} (no variation or empty after dropna)")
            continue

        if model_type == "logit":
            print(f"\n[{label}] {outcome} distribution:")
            print(pd.crosstab(df_sub["group"], df_sub[outcome], dropna=False))
        else:
            print(f"\n[{label}] {outcome} summary:")
            print(df_sub.groupby("group", observed=False)[outcome].describe())
        print(f"[{label}] reference group = {ref_group} | covariates = {covariates}")

        formula = f"{outcome} ~ " + " + ".join([group_term] + covariates)
        try:
            model = (smf.logit(formula, data=df_sub).fit(disp=False) if model_type == "logit"
                     else smf.ols(formula, data=df_sub).fit())
            all_results.append(extract_adjusted_results(model, outcome, label, model_type))
        except Exception as e:
            print(f"[{label}] Error in {outcome}: {e}")

    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

In [ ]:
all_outputs = []
for label, filename in MATCHED_FILES.items():
    print(f"\n========== Running {label} ==========")
    res = run_minimal_models(
        df=pd.read_csv(output_dir / filename),
        label=label,
        ref_group=REFERENCE_MAP[label],
        treat_group=TREAT_MAP.get(label))
    if not res.empty:
        all_outputs.append(res)

if all_outputs:
    final_df = pd.concat(all_outputs, ignore_index=True)
    group_effect_only = final_df[
        final_df["variable"].str.contains("group", case=False, na=False)].copy()
else:
    final_df = group_effect_only = pd.DataFrame()

print("\n===== MINIMALLY ADJUSTED GROUP EFFECT =====")
if not group_effect_only.empty:
    print(group_effect_only[["comparison", "outcome", "variable",
                             "formatted", "model_type", "p_value"]])
else:
    print("No group effect rows found.")

group_effect_only.to_csv(output_dir / "unadjusted_summary_all.csv", index=False)
group_effect_only.to_csv(output_dir / "adjusted_summary_all.csv", index=False)

## 11. Table 1

In [ ]:
TABLE1_COLS = [
    "pregnancy_exposure_group", "user_group",
    "age_at_delivery", "race_ethnicity", "Income", "parity",
    "preTreatmentBMI", "PrePregnancyBMI", "weight_loss",
    "med_indict", "AccumulatedPersistenceBeforePregnancy", "TotalExposureDaysInPregnancy",
    "t2d_before_pregnancy", "hyper_before_pregnancy", "depression",
    "prior_Csection", "Prior_Preterm_Birth", "supply_days", "obstetric_care",
    "zcode_count", "gestational_week", "gestation_weight", "gwg_category",
    "gest_diabetes_no_prior_t2d", "preg_related_htn", "excessive_fetal_weight",
    "intra_grow_restrict", "csection", "preterm",
]

TABLE1_CATEGORICAL = [
    "pregnancy_exposure_group", "user_group", "race_ethnicity", "Income", "parity",
    "med_indict", "t2d_before_pregnancy", "hyper_before_pregnancy", "depression",
    "prior_Csection", "Prior_Preterm_Birth", "obstetric_care", "gwg_category",
    "gest_diabetes_no_prior_t2d", "preg_related_htn", "excessive_fetal_weight",
    "intra_grow_restrict", "csection", "preterm",
]

# Reported as median [Q1, Q3] rather than mean (SD).
TABLE1_NONNORMAL = [
    "AccumulatedPersistenceBeforePregnancy", "TotalExposureDaysInPregnancy",
    "supply_days", "gestational_week", "weight_loss",
]

### 11.1 Matched sets

In [ ]:
for label, filename in MATCHED_FILES.items():
    print(f"\n===== {label} =====")
    df_t1 = pd.read_csv(output_dir / filename)

    if "user_group" not in df_t1.columns:
        print(f"Skipped {label}: user_group not found.")
        continue
    df_t1["user_group"] = df_t1["user_group"].astype(str)

    available_cols = [c for c in TABLE1_COLS if c in df_t1.columns]
    available_categorical = [c for c in TABLE1_CATEGORICAL if c in df_t1.columns]
    available_nonnormal = [c for c in TABLE1_NONNORMAL if c in df_t1.columns]

    missing_cols = [c for c in TABLE1_COLS if c not in df_t1.columns]
    if missing_cols:
        print("Missing columns skipped:", missing_cols)

    for c in available_nonnormal:
        df_t1[c] = pd.to_numeric(df_t1[c], errors="coerce")

    table1 = TableOne(data=df_t1, columns=available_cols,
                      categorical=available_categorical, groupby="user_group",
                      nonnormal=available_nonnormal, pval=True)
    table1.to_csv(output_dir / f"{label}_table1_tableone.csv")
    print(table1)

### 11.2 Unmatched cohorts

In [ ]:
UNMATCHED_COLS = [c for c in TABLE1_COLS if c != "pregnancy_exposure_group"]
UNMATCHED_CATEGORICAL = [c for c in TABLE1_CATEGORICAL if c != "pregnancy_exposure_group"]
UNMATCHED_NONNORMAL = [c for c in TABLE1_NONNORMAL if c != "supply_days"]

df_all["user_group"] = df_all["user_group"].astype(str)
for c in UNMATCHED_NONNORMAL:
    if c in df_all.columns:
        df_all[c] = pd.to_numeric(df_all[c], errors="coerce")

table1_unmatch = TableOne(data=df_all, columns=UNMATCHED_COLS,
                          categorical=UNMATCHED_CATEGORICAL, groupby="user_group",
                          nonnormal=UNMATCHED_NONNORMAL, pval=True)
table1_unmatch.to_csv(output_dir / "unmatched_table1_tableone.csv")
print(table1_unmatch)

In [ ]:
# Pairwise, unmatched.
df_cf = df_all[df_all["user_group"].isin(["continued_user", "former_user"])].copy()
table1_cf = TableOne(data=df_cf, columns=UNMATCHED_COLS,
                     categorical=UNMATCHED_CATEGORICAL, groupby="user_group",
                     nonnormal=UNMATCHED_NONNORMAL, pval=True)
table1_cf.to_csv(output_dir / "um_table1_continued_vs_former.csv")
table1_cf

In [ ]:
df_cn = df_all[df_all["user_group"].isin(["continued_user", "non_user"])].copy()
table1_cn = TableOne(data=df_cn, columns=UNMATCHED_COLS,
                     categorical=UNMATCHED_CATEGORICAL, groupby="user_group",
                     nonnormal=UNMATCHED_NONNORMAL, pval=True)
table1_cn.to_csv(output_dir / "table1_continued_vs_non.csv")
table1_cn

### 11.3 Continued users, by trimester exposure pattern

In [ ]:
df_preg = df_preg.copy()
df_preg["pregnancy_exposure_group"] = df_preg["pregnancy_exposure_group"].astype(str)
for c in TABLE1_NONNORMAL:
    if c in df_preg.columns:
        df_preg[c] = pd.to_numeric(df_preg[c], errors="coerce")

preg_cols = [c for c in TABLE1_COLS + ["new_refill"] if c in df_preg.columns]
preg_categorical = [c for c in TABLE1_CATEGORICAL + ["new_refill"] if c in df_preg.columns]

table1_preg = TableOne(data=df_preg, columns=preg_cols, categorical=preg_categorical,
                       groupby="pregnancy_exposure_group",
                       nonnormal=[c for c in TABLE1_NONNORMAL if c in df_preg.columns],
                       pval=True)
table1_preg.to_csv(output_dir / "pregnancy_exposure_group_table1_tableone.csv")
table1_preg

## 12. Post hoc analysis of the GWG category distribution

For each matched comparison: an overall chi-square test on the 2 x 3
(group x GWG category) table, standardised residuals to show which cells drive
it, and Holm-adjusted two-proportion z-tests per category.

In [ ]:
def analyze_gwg_table(df, group_col="user_group", gwg_col="gwg_category", alpha=0.05):
    """Chi-square plus per-category post hoc tests for a 2 x K contingency table."""
    dat = df[[group_col, gwg_col]].dropna().copy()
    dat[group_col] = dat[group_col].astype(str)
    dat[gwg_col] = dat[gwg_col].astype(str)

    ct = pd.crosstab(dat[group_col], dat[gwg_col], dropna=False)
    if ct.shape[0] != 2:
        raise ValueError(
            f"Expected exactly 2 groups in '{group_col}', found {ct.shape[0]}: {list(ct.index)}")

    chi2, p_value, dof, expected = chi2_contingency(ct)
    expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)

    std_resid = pd.DataFrame(Table(ct.values).standardized_resids,
                             index=ct.index, columns=ct.columns)

    g1, g2 = list(ct.index)
    n1, n2 = ct.loc[g1].sum(), ct.loc[g2].sum()

    posthoc_rows = []
    for category in ct.columns:
        count1, count2 = int(ct.loc[g1, category]), int(ct.loc[g2, category])
        z_stat, p_raw = proportions_ztest(count=np.array([count1, count2]),
                                          nobs=np.array([n1, n2]),
                                          alternative="two-sided")
        prop1 = count1 / n1 if n1 > 0 else np.nan
        prop2 = count2 / n2 if n2 > 0 else np.nan

        posthoc_rows.append({
            "gwg_category": category,
            "group_1": g1, "count_1": count1, "n_1": n1, "prop_1": prop1,
            "group_2": g2, "count_2": count2, "n_2": n2, "prop_2": prop2,
            "risk_difference": prop1 - prop2, "z_stat": z_stat, "p_raw": p_raw,
        })

    posthoc_df = pd.DataFrame(posthoc_rows)
    reject, p_holm, _, _ = multipletests(posthoc_df["p_raw"], alpha=alpha, method="holm")
    posthoc_df["p_holm"] = p_holm
    posthoc_df["significant_holm"] = reject

    resid_flags = std_resid.applymap(
        lambda x: ">|2.58|" if abs(x) > 2.58 else (">|1.96|" if abs(x) > 1.96 else ""))

    chi2_results = {
        "chi2": chi2, "dof": dof, "p_value": p_value,
        "n_total": int(ct.values.sum()),
        "min_expected": float(expected_df.min().min()),
        "cells_expected_lt_5": int((expected_df < 5).sum().sum()),
        "significant": bool(p_value < alpha),
    }

    return {"contingency_table": ct, "expected_counts": expected_df,
            "chi2_results": chi2_results, "standardized_residuals": std_resid,
            "residual_flags": resid_flags, "category_posthoc": posthoc_df}

In [ ]:
gwg_results = {}

for comparison_name, filename in MATCHED_FILES.items():
    result = analyze_gwg_table(pd.read_csv(output_dir / filename))
    gwg_results[comparison_name] = result

    print("\n" + "=" * 80)
    print(f"Comparison: {comparison_name}")
    print("=" * 80)

    print("\nContingency table")
    print(result["contingency_table"])

    print("\nExpected counts")
    print(result["expected_counts"].round(2))

    print("\nOverall chi-square")
    for k, v in result["chi2_results"].items():
        print(f"{k}: {v}")

    print("\nStandardized residuals")
    print(result["standardized_residuals"].round(3))
    print("\nResidual flags")
    print(result["residual_flags"])

    print("\nCategory-specific post hoc tests (Holm-adjusted)")
    print(result["category_posthoc"][[
        "gwg_category", "group_1", "count_1", "n_1", "prop_1",
        "group_2", "count_2", "n_2", "prop_2",
        "risk_difference", "z_stat", "p_raw", "p_holm", "significant_holm"]].round(4))

In [ ]:
pd.DataFrame([{
    "comparison": name,
    "n_total": r["chi2_results"]["n_total"],
    "chi2": r["chi2_results"]["chi2"],
    "dof": r["chi2_results"]["dof"],
    "p_value": r["chi2_results"]["p_value"],
    "min_expected": r["chi2_results"]["min_expected"],
    "cells_expected_lt_5": r["chi2_results"]["cells_expected_lt_5"],
} for name, r in gwg_results.items()]).round(4)

In [ ]:
csv_dir = output_dir / "csv_outputs"
csv_dir.mkdir(parents=True, exist_ok=True)

for comparison_name, result in gwg_results.items():
    # Counts with row percentages, ready to paste into a manuscript table.
    ct = result["contingency_table"].copy()
    row_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_formatted = (ct.astype(str) + " (" + row_pct.round(1).astype(str) + "%)").reset_index()

    posthoc = result["category_posthoc"].copy()
    for c in ["prop_1", "prop_2", "risk_difference"]:
        if c in posthoc.columns:
            posthoc[c] = (posthoc[c] * 100).round(1)

    ct_formatted.to_csv(csv_dir / f"{comparison_name}_table3.csv", index=False)
    result["expected_counts"].round(2).reset_index().to_csv(
        csv_dir / f"{comparison_name}_expected.csv", index=False)
    pd.DataFrame([result["chi2_results"]]).to_csv(
        csv_dir / f"{comparison_name}_chi2.csv", index=False)
    posthoc.to_csv(csv_dir / f"{comparison_name}_posthoc.csv", index=False)

print("All CSV files saved to:", csv_dir)

## 13. Sensitivity analysis: people excluded for missing weight data

Requiring both a pre-pregnancy and a pre-delivery weight removes a large share
of otherwise eligible deliveries. This section compares those who were kept
(`With_Weight`) against those who were dropped (`Without_Weight`) on the
covariates and outcomes that do **not** depend on weight, to gauge how selective
that requirement is.

In [ ]:
full_control_df = pd.read_csv(output_path_local + IN_CONTROL_NOWT)
full_med_df = pd.read_csv(output_path_local + IN_EXPOSED_NOWT)

df_control_sub["weights"] = "With_Weight"
full_control_df["weights"] = "Without_Weight"
df_treated_sub["weights"] = "With_Weight"
full_med_df["weights"] = "Without_Weight"

In [ ]:
# Harmonise the columns the no-weight exports carry under different names.
if "obstetric_care" not in full_control_df.columns and "Obstetriccare" in full_control_df.columns:
    full_control_df = full_control_df.rename(columns={"Obstetriccare": "obstetric_care"})

for frame in (full_med_df, full_control_df):
    frame["preg_related_htn"] = (
        (frame["gest_hyper_no_prior_hyper"] == True) |
        (frame["preeclampsia_no_prior_hyper"] == True))

contrl_zcodecount = pd.read_csv(output_path_local + IN_ZCODE_CONTROL)
full_control_df = full_control_df.merge(contrl_zcodecount, on="PersonId")

In [ ]:
SENSITIVITY_COLS = [
    "weights",
    "age_at_delivery", "race_ethnicity", "Income", "parity",
    "t2d_before_pregnancy", "hyper_before_pregnancy", "depression",
    "prior_Csection", "Prior_Preterm_Birth", "obstetric_care",
    "zcode_count", "gestational_week",
    "gest_diabetes_no_prior_t2d", "preg_related_htn", "excessive_fetal_weight",
    "intra_grow_restrict", "csection", "preterm",
]

SENSITIVITY_CATEGORICAL = [
    "weights", "race_ethnicity", "Income", "parity",
    "t2d_before_pregnancy", "hyper_before_pregnancy", "depression",
    "prior_Csection", "Prior_Preterm_Birth", "obstetric_care",
    "gest_diabetes_no_prior_t2d", "preg_related_htn", "excessive_fetal_weight",
    "intra_grow_restrict", "csection", "preterm",
]

SENSITIVITY_NONNORMAL = ["gestational_week"]

control_w = pd.concat([full_control_df[SENSITIVITY_COLS],
                       df_control_sub[SENSITIVITY_COLS]], ignore_index=True)
med_w = pd.concat([full_med_df[SENSITIVITY_COLS],
                   df_treated_sub[SENSITIVITY_COLS]], ignore_index=True)

In [ ]:
for frame in (control_w, med_w):
    frame["weights"] = frame["weights"].astype(str)
    for c in SENSITIVITY_NONNORMAL:
        frame[c] = pd.to_numeric(frame[c], errors="coerce")

table_control_w = TableOne(data=control_w, columns=SENSITIVITY_COLS,
                           categorical=SENSITIVITY_CATEGORICAL,
                           groupby="weights", nonnormal=SENSITIVITY_NONNORMAL, pval=True)
table_control_w.to_csv(output_dir / "controlw_table1_tableone.csv")
table_control_w

In [ ]:
table_med_w = TableOne(data=med_w, columns=SENSITIVITY_COLS,
                       categorical=SENSITIVITY_CATEGORICAL,
                       groupby="weights", nonnormal=SENSITIVITY_NONNORMAL, pval=True)
table_med_w.to_csv(output_dir / "med_w_table1_tableone.csv")
table_med_w

---

## Appendix - superseded approaches (do not run)

### A1. GWG category from the last in-pregnancy weight

An alternative GWG definition that recomputed the weight gain from the last
weight measured within 28 days of delivery, rather than from
`gestation_weight`. Superseded by section 5, which uses the cohort-level
`gestation_weight` and gestational age at delivery.

In [ ]:
# --- SUPERSEDED - DO NOT RUN ---
# weights_test = pd.read_csv(output_path_local + "/weights_test.csv")
#
# weight_df = weights_test.merge(df[["PersonId", "delivery_date", "estimated_LMP"]],
#                                on="PersonId", how="inner")
# for c in ["RecordedDateTime", "delivery_date", "estimated_LMP"]:
#     weight_df[c] = pd.to_datetime(weight_df[c], errors="coerce")
#
# # Keep in-pregnancy weights measured in the last 28 days before delivery.
# weight_df = weight_df[(weight_df["RecordedDateTime"] >= weight_df["estimated_LMP"]) &
#                       (weight_df["RecordedDateTime"] <= weight_df["delivery_date"])]
# weight_df["diff_days"] = (weight_df["delivery_date"] - weight_df["RecordedDateTime"]).dt.days
# weight_df = weight_df[(weight_df["diff_days"] >= 0) & (weight_df["diff_days"] <= 28)]
#
# idx = weight_df.groupby(["PersonId", "delivery_date"])["RecordedDateTime"].idxmax()
# last_weight_df = weight_df.loc[idx].copy()
#
# analysis_df = last_weight_df.merge(
#     df[["PersonId", "prepreg_weight", "PrePregnancyBMI", "user_group"]],
#     on="PersonId", how="left")
# analysis_df["GWG"] = analysis_df["last_weight_kg"] - analysis_df["prepreg_weight"]
# # ... then bmi_category / gwg_bounds / categorize_gwg as in section 5, using
# # `gestational_week_last_weight` in place of `gestational_week`, restricted to
# # 24-42 weeks.

### A2. Rebuilding `obstetric_care` and C-section for the control cohort

At one point the control export was missing these two flags and they were
rebuilt here. Notebook 02 now produces both, so this is no longer needed.

In [ ]:
# --- SUPERSEDED - notebook 02 produces these - DO NOT RUN ---
# Prenatal, _ = load_condition_data(snapshot, codeset_url="/definitions/prenatal",
#                                   codes="Prenatal", view_name="tbl_index_Prenatal")
# csection_code = snapshot.codeset_from_prose(url="/definitions/c-section", variable_name="codes")
# ...
# control_df = control_df.drop(columns=["obstetric_care", "csection1"])
# control_df = mark_condition_in_pregnancy(Prenatal, control_df, "obstetric_care")
# control_df = mark_condition_in_pregnancy(csection, control_df, "csection1")